# Evaluación del modelo y variantes

En el capítulo anterior elegimos XGBoost, pero lo medimos con pocas métricas y con una sola
separación de zonas. Aquí calculamos todas las métricas necesarias y repetimos la separación de
zonas para saber cuánto depende el resultado del azar. Con eso probamos tres cambios que no
necesitan datos nuevos (el barrio como variable, modelos separados para casas y apartamentos, y
otra función de pérdida), agregamos el área del lote de las casas y revisamos otras formas de
optimizar el modelo.

Las variantes se comparan solo dentro del conjunto de entrenamiento. La prueba final se usa una
sola vez, al terminar, con la variante ya elegida.

In [1]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "4")   # evita que XGBoost y scikit-learn se bloqueen en Jupyter
from eda_utils import *
import time
import xgboost as xgb
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupKFold, ParameterSampler
from sklearn.neighbors import BallTree
from sklearn.pipeline import make_pipeline
from xgboost import XGBRegressor
from sklearn.model_selection import KFold

In [2]:
d = viv[viv.lat.notna() & viv.lon.notna() & viv.area.notna()].copy().reset_index(drop=True)
a_2022 = ipc_anio.loc[2022] / ipc_anio
d["razon_avaluo"] = d.precio * d.anio.map(a_2022) / d.avaluo
d["al_avaluo"] = d.razon_avaluo.between(0.9, 1.1).fillna(False).astype(bool)   # sin avalúo: no se puede saber
d["tipo_txt"] = np.where((d.condicion_predio == "NPH").fillna(False), "casa", "apartamento u otro")

d = d[~d.al_avaluo].reset_index(drop=True)
d["log_area"] = np.log10(d.area)
d["log_pm2"] = np.log10(d.precio_m2_real)
d["estrato_n"] = d.estrato.astype(str).map({e: i + 1 for i, e in enumerate(ESTRATOS)})
for t in ["PH_Unidad_Predial", "NPH", "PH_Matriz"]:
    d[f"tipo_{t}"] = (d.condicion_predio == t).astype(float)
for f in ["igac", "observatorio", "georreferenciadas"]:
    d[f"fuente_{f}"] = (d.fuente == f).astype(float)
d["bloque"] = (np.floor(d.lat / 0.01).astype(int).astype(str) + "_" +
               np.floor(d.lon / 0.01).astype(int).astype(str))

BASE = ["log_area", "habitaciones", "banios", "piso", "plantas", "edad", "estrato_n", "lat", "lon",
        "anio", "tipo_PH_Unidad_Predial", "tipo_NPH", "tipo_PH_Matriz",
        "fuente_igac", "fuente_observatorio", "fuente_georreferenciadas"]

bloques = d.groupby("bloque").agg(ventas=("bloque", "size"),
                                  alameda=("barrio", lambda s: (s == "SECTOR ALAMEDA DEL RIO").mean()))
candidatos = bloques[bloques.alameda < 0.5].sample(frac=1, random_state=42)
bloques_prueba = candidatos.index[candidatos.ventas.cumsum() <= 0.2 * len(d)]
es_prueba = d.bloque.isin(bloques_prueba)
ent, prueba = d[~es_prueba].reset_index(drop=True), d[es_prueba].reset_index(drop=True)
y_ent, y_prueba = ent.log_precio.to_numpy(), prueba.log_precio.to_numpy()
print(f"Entrenamiento: {len(ent):,} ventas en {ent.bloque.nunique()} bloques")
print(f"Prueba final : {len(prueba):,} ventas en {prueba.bloque.nunique()} bloques")

Entrenamiento: 11,324 ventas en 71 bloques
Prueba final : 2,400 ventas en 24 bloques


## Las métricas

| Métrica | Qué mide | Mejor si | Referencia |
|---|---|---|---|
| **R² (log)** | Parte de la variación del log del precio que explica el modelo | Cerca de 1 | |
| **R² (pesos)** | Lo mismo en pesos; lo dominan las viviendas más caras | Cerca de 1 | |
| **RMSE y MAE (log10)** | Error en la escala en que se entrena | Bajos | |
| **RMSE, MAE y error absoluto mediano (millones)** | Error en pesos de diciembre de 2025 | Bajos | |
| **MdAPE** | Error porcentual mediano: la mitad de las ventas erra menos que esto | Bajo | |
| **MAPE** | Error porcentual promedio; le pesan los errores grandes | Bajo | |
| **P90 del error** | El 10% de las ventas erra más que esto | Bajo | |
| **PPE10, PPE20, PPE30** | Porcentaje de ventas con error menor a 10%, 20% y 30% | Altos | |
| **Sesgo mediano** | Negativo: el modelo predice menos de lo pagado | Cerca de 0 | |
| **Razón mediana** | Mediana de predicho / pagado | Cerca de 1 | IAAO: 0,90 a 1,10 |
| **COD** | Dispersión de esa razón alrededor de su mediana | Bajo | IAAO: 5 a 15 en residencial |
| **PRD** | Mayor que 1: el modelo sobrevalora las baratas frente a las caras | Cerca de 1 | IAAO: 0,98 a 1,03 |
| **PRB** | Cambio de la razón cuando el valor se duplica; negativo indica lo mismo que un PRD alto | Cerca de 0 | IAAO: −0,05 a 0,05 |
| **I de Moran de los residuos** | Si los errores de vecinas se parecen: queda información espacial sin capturar | Cerca de 0 | |

Las referencias son de la IAAO (*Standard on Ratio Studies*, 2013). Esa norma evalúa avalúos
masivos contra ventas ya depuradas, así que aquí sirve como guía y no como un examen que el
modelo deba aprobar. La norma europea y la RICS no fijan un valor aprobatorio.

In [3]:
R_TIERRA_KM = 6371

def moran(residuo, lat, lon, k=8):
    """I de Moran con las k ventas más cercanas."""
    xy = np.radians(np.c_[lat, lon])
    idx = BallTree(xy, metric="haversine").query(xy, k=k + 1)[1]
    otras = idx != np.arange(len(xy))[:, None]        # quita la propia venta aunque haya coordenadas repetidas
    otras[otras.all(axis=1), -1] = False
    e = residuo - residuo.mean()
    return np.nansum(e * np.nanmean(np.where(otras, e[idx], np.nan), axis=1)) / (e ** 2).sum()

def metricas_completas(y, p, lat=None, lon=None):
    real, pred = 10.0 ** y, 10.0 ** p
    err = pred / real - 1
    ae = np.abs(err)
    r = pred / real
    med = np.median(r)
    valor = 0.5 * (pred / med + real)                 # PRB según IAAO
    m = {"R² (log)": 1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum(),
         "R² (pesos)": 1 - ((real - pred) ** 2).sum() / ((real - real.mean()) ** 2).sum(),
         "RMSE (log10)": np.sqrt(np.mean((y - p) ** 2)),
         "MAE (log10)": np.mean(np.abs(y - p)),
         "RMSE (millones)": np.sqrt(np.mean((real - pred) ** 2)) / 1e6,
         "MAE (millones)": np.mean(np.abs(real - pred)) / 1e6,
         "Error abs. mediano (millones)": np.median(np.abs(real - pred)) / 1e6,
         "MdAPE (%)": np.median(ae) * 100,
         "MAPE (%)": ae.mean() * 100,
         "P90 del error (%)": np.quantile(ae, 0.9) * 100,
         "PPE10 (%)": (ae <= 0.1).mean() * 100,
         "PPE20 (%)": (ae <= 0.2).mean() * 100,
         "PPE30 (%)": (ae <= 0.3).mean() * 100,
         "Sesgo mediano (%)": np.median(err) * 100,
         "Razón mediana": med,
         "COD": 100 * np.mean(np.abs(r - med)) / med,
         "PRD": r.mean() / (pred.sum() / real.sum()),
         "PRB": np.polyfit(np.log2(valor), (r - med) / med, 1)[0]}
    if lat is not None:
        m["I de Moran (residuos)"] = moran(y - p, lat, lon)
    return m

## Variantes y diseño de la comparación

Partimos de XGBoost con los hiperparámetros del capítulo anterior y probamos:

- **Barrio:** precio por m² del barrio, suavizado. En entrenamiento se calcula con ventas de otros
  bloques, para que el modelo lo vea como lo verá en una zona nueva. Si el barrio no aparece en
  entrenamiento queda vacío.
- **Separado:** un modelo para casas y otro para lo demás.
- **Pérdida MAE o Huber:** cambian el error cuadrático por pérdidas que se ajustan mejor a la
  mediana y les pesan menos los precios atípicos. A Huber hay que darle el valor inicial, porque
  XGBoost lo estima mal con esta pérdida.

Como referencia incluimos la regresión lineal y una regla ingenua: el precio mediano por m² del
estrato multiplicado por el área.

Cada variante se evalúa en 20 divisiones por zonas del conjunto de entrenamiento. Todas usan las
mismas 20 divisiones, de modo que las diferencias se pueden medir división por división. Cada una
deja aparte cerca del 20% de las ventas. El bloque de Alameda del Río se queda siempre en
entrenamiento, igual que en la prueba final.

In [4]:
MEJORES = dict(n_estimators=600, learning_rate=0.05, max_depth=6, subsample=0.6,
               colsample_bytree=0.8, min_child_weight=30, reg_lambda=10.0)
MAE = dict(objective="reg:absoluteerror")
HUBER = dict(objective="reg:pseudohubererror", huber_slope=0.1)
CON_BARRIO = BASE + ["enc_barrio"]

def ajustar(tr, te, columnas, **extra):
    if extra.get("objective") == "reg:pseudohubererror":
        extra = {**extra, "base_score": float(tr.log_precio.median())}
    modelo = XGBRegressor(n_jobs=4, random_state=42, **{**MEJORES, **extra})
    return modelo.fit(tr[columnas], tr.log_precio).predict(te[columnas]).astype(float)

def separar(tr, te, columnas, **extra):
    p = np.empty(len(te))
    for es_casa in [True, False]:
        mascara = (te.tipo_NPH == 1).to_numpy() == es_casa
        if mascara.any():
            p[mascara] = ajustar(tr[(tr.tipo_NPH == 1) == es_casa], te[mascara], columnas, **extra)
    return p

def con_barrio(tr, te, m=20):
    tr, te = tr.copy(), te.copy()
    media = tr.log_pm2.mean()
    def mapa(a):
        t = a.groupby("barrio").log_pm2.agg(["mean", "size"])
        return (t["mean"] * t["size"] + media * m) / (t["size"] + m)
    enc = np.full(len(tr), np.nan)
    for a, b in GroupKFold(5).split(tr, groups=tr.bloque):
        enc[b] = tr.iloc[b].barrio.map(mapa(tr.iloc[a])).astype(float).to_numpy()
    tr["enc_barrio"], te["enc_barrio"] = enc, te.barrio.map(mapa(tr)).astype(float).to_numpy()
    return tr, te

def ingenua(tr, te):
    por_estrato = tr.groupby("estrato_n").log_pm2.median()
    return te.estrato_n.map(por_estrato).fillna(tr.log_pm2.median()).to_numpy() + te.log_area.to_numpy()

def lineal(tr, te):
    f = make_pipeline(SimpleImputer(strategy="median", add_indicator=True), LinearRegression())
    return f.fit(tr[BASE], tr.log_precio).predict(te[BASE])

VARIANTES = {
    "Regla ingenua": ingenua,
    "Regresión lineal": lineal,
    "XGBoost actual": lambda a, b: ajustar(a, b, BASE),
    "+ barrio": lambda a, b: ajustar(*con_barrio(a, b), CON_BARRIO),
    "Separado": lambda a, b: separar(a, b, BASE),
    "Pérdida MAE": lambda a, b: ajustar(a, b, BASE, **MAE),
    "Pérdida Huber": lambda a, b: ajustar(a, b, BASE, **HUBER),
    "Huber, min_child_weight 10": lambda a, b: ajustar(a, b, BASE, **{**HUBER, "min_child_weight": 10}),
    "+ barrio, separado": lambda a, b: separar(*con_barrio(a, b), CON_BARRIO),
    "+ barrio, MAE": lambda a, b: ajustar(*con_barrio(a, b), CON_BARRIO, **MAE),
    "+ barrio, Huber": lambda a, b: ajustar(*con_barrio(a, b), CON_BARRIO, **HUBER),
    "+ barrio, separado, Huber": lambda a, b: separar(*con_barrio(a, b), CON_BARRIO, **HUBER),
}

def divisiones(datos, n=20, frac=0.2):
    bl = datos.groupby("bloque").agg(ventas=("bloque", "size"),
                                     alameda=("barrio", lambda s: (s == "SECTOR ALAMEDA DEL RIO").mean()))
    candidatos = bl[bl.alameda < 0.5]
    for s in range(n):
        c = candidatos.sample(frac=1, random_state=s)
        yield s, datos.bloque.isin(c.index[c.ventas.cumsum() <= frac * len(datos)]).to_numpy()

filas = []
for s, es_val in divisiones(ent):
    tr, te = ent[~es_val].reset_index(drop=True), ent[es_val].reset_index(drop=True)
    y = te.log_precio.to_numpy()
    casa = (te.tipo_NPH == 1).to_numpy()
    sin_barrio = te.barrio.map(tr.barrio.value_counts()).isna().mean() * 100
    for nombre, f in VARIANTES.items():
        p = f(tr, te)
        filas.append({"division": s, "modelo": nombre, "grupo": "todas", "ventas": len(te), "sin_barrio": sin_barrio,
                      **metricas_completas(y, p, te.lat.to_numpy(), te.lon.to_numpy())})
        for grupo, mascara in [("casas", casa), ("apartamentos u otros", ~casa)]:
            if mascara.sum() > 30:
                filas.append({"division": s, "modelo": nombre, "grupo": grupo, "ventas": int(mascara.sum()),
                              **metricas_completas(y[mascara], p[mascara])})
resultados = pd.DataFrame(filas)
todas = resultados[resultados.grupo == "todas"]
orden = todas.groupby("modelo")["MdAPE (%)"].mean().sort_values().index
METRICAS = [c for c in resultados.columns if c not in ("division", "modelo", "grupo", "ventas", "sin_barrio")]
todas.groupby("modelo")[METRICAS].mean().loc[orden].T.round(2)

modelo,Separado,Pérdida MAE,"+ barrio, MAE","+ barrio, separado",+ barrio,Pérdida Huber,XGBoost actual,"Huber, min_child_weight 10","+ barrio, separado, Huber","+ barrio, Huber",Regresión lineal,Regla ingenua
R² (log),0.63,0.63,0.63,0.63,0.63,0.63,0.62,0.62,0.63,0.62,0.57,0.35
R² (pesos),0.36,0.36,0.36,0.37,0.37,0.36,0.36,0.36,0.35,0.37,-5.26,-22.30
RMSE (log10),0.20,0.20,0.20,0.20,0.20,0.20,0.20,0.20,0.20,0.20,0.22,0.27
MAE (log10),0.15,0.15,0.15,0.15,0.15,0.15,0.15,0.15,0.15,0.15,0.16,0.19
RMSE (millones),368.45,369.35,367.50,366.25,364.94,367.95,366.35,366.06,373.80,366.99,576.69,"2,055.15"
MAE (millones),125.01,122.37,122.34,125.00,123.12,122.16,123.31,122.27,126.27,122.64,137.49,210.81
Error abs. mediano (millones),58.16,58.23,58.63,59.52,58.60,59.27,59.26,59.34,59.38,59.06,63.77,80.24
MdAPE (%),24.33,24.42,24.72,24.85,25.01,25.13,25.15,25.16,25.33,25.42,26.59,31.04
MAPE (%),36.61,36.42,36.80,37.02,36.86,37.03,37.04,36.86,37.01,37.37,43.85,67.25
P90 del error (%),72.97,72.60,73.21,73.47,72.86,73.04,72.03,72.40,73.56,73.60,82.76,161.89


Tabla: promedio de cada métrica en las 20 divisiones, con las variantes de menor a mayor error
mediano.

XGBoost supera con claridad a las dos referencias: la regla ingenua tiene un error mediano de 31,0%
y la regresión lineal de 26,6%, frente a 25,1% de XGBoost. Pero aún está lejos de las referencias
de la IAAO: el COD ronda 38 (referencia: hasta 15) y el PRD 1,24 (referencia: hasta 1,03). El modelo
sobrevalora las viviendas baratas y subvalora las caras. El I de Moran de 0,21 indica que los errores
de viviendas cercanas todavía se parecen.

## ¿Qué tanto depende del azar?

In [5]:
xgb_actual = todas[todas.modelo == "XGBoost actual"]
xgb_actual[["MdAPE (%)", "R² (log)", "Sesgo mediano (%)", "ventas"]].describe().loc[["mean", "std", "min", "max"]].round(2)

,MdAPE (%),R² (log),Sesgo mediano (%),ventas
mean,25.15,0.62,-6.05,"2,035.15"
std,4.78,0.09,6.03,241.92
min,15.50,0.31,-18.22,"1,342.00"
max,31.86,0.75,2.01,"2,259.00"


Es el mismo modelo con los mismos datos, y su error mediano va de 15,5% a 31,9% según qué zonas
queden aparte (desviación de 4,8 puntos). También el sesgo cambia de signo, de −18% a +2%. Por eso
una sola separación no alcanza para concluir nada, y el sesgo de −10% de la prueba final puede
deberse en buena parte a las zonas que cayeron ahí.

## ¿Qué variante gana?

Para cada variante calculamos, división por división, la diferencia de error mediano con XGBoost
actual. El intervalo es aproximado porque las divisiones se traslapan, y al comparar varias
variantes alguna puede salir favorecida por azar.

In [6]:
por_division = todas.pivot(index="division", columns="modelo", values="MdAPE (%)")
dif = por_division.sub(por_division["XGBoost actual"], axis=0).drop(columns="XGBoost actual")
tabla_dif = pd.DataFrame({"diferencia media": dif.mean(),
                          "IC95 inf": dif.mean() - 2.093 * dif.std() / np.sqrt(len(dif)),
                          "IC95 sup": dif.mean() + 2.093 * dif.std() / np.sqrt(len(dif)),
                          "divisiones en que mejora": (dif < 0).sum()}).sort_values("diferencia media")
tabla_dif.round(2)

,diferencia media,IC95 inf,IC95 sup,divisiones en que mejora
modelo,,,,
Separado,-0.82,-2.43,0.80,9
Pérdida MAE,-0.73,-1.32,-0.15,15
"+ barrio, MAE",-0.43,-1.23,0.38,12
"+ barrio, separado",-0.30,-1.74,1.14,7
+ barrio,-0.14,-0.59,0.31,9
Pérdida Huber,-0.02,-0.67,0.63,13
"Huber, min_child_weight 10",0.01,-0.62,0.65,12
"+ barrio, separado, Huber",0.19,-0.95,1.32,9
"+ barrio, Huber",0.27,-0.61,1.16,11


In [7]:
(resultados[resultados.grupo != "todas"]
    .pivot_table(index="modelo", columns="grupo", values=["MdAPE (%)", "Sesgo mediano (%)"])
    .loc[orden].round(1))

MdAPE (%)          Sesgo mediano (%)      
grupo                      apartamentos u otros casas apartamentos u otros casas
modelo                                                                          
Separado                                  21.30 32.50                -5.00  0.10
Pérdida MAE                               21.90 31.50                -7.30  1.40
+ barrio, MAE                             22.00 31.60                -7.10  2.00
+ barrio, separado                        22.10 32.80                -4.90  1.00
+ barrio                                  22.50 31.80                -7.50  0.90
Pérdida Huber                             22.80 31.20                -8.10  1.90
XGBoost actual                            23.00 31.10                -8.50  0.70
Huber, min_child_weight 10                23.10 31.10                -9.40  1.40
+ barrio, separado, Huber                 22.80 32.60                -6.20  1.90
+ barrio, Huber                           23.00 31.70                -7.30  2.70
Regresión lineal                          23.00 35.10                -4.20  0.20
Regla ingenua                             25.00 58.30                -2.60 53.90

In [8]:
print(f"Ventas de validación en un barrio que no aparece en entrenamiento: {todas.sin_barrio.mean():.0f}% en promedio")

Ventas de validación en un barrio que no aparece en entrenamiento: 43% en promedio


Ninguna variante cambia mucho el resultado:

- **Pérdida MAE:** es la única cuyo intervalo no incluye el cero. Baja el error mediano 0,7 puntos y
  mejora en 15 de las 20 divisiones. Es poco, pero es consistente.
- **Modelos separados:** bajan en promedio 0,8 puntos, pero mejoran solo en 9 de 20 divisiones.
  Ayudan a los apartamentos (de 23,0% a 21,3%) y empeoran a las casas (de 31,1% a 32,5%).
- **Barrio:** no aporta. En promedio, el 43% de las ventas de validación está en un barrio que no
  aparece en entrenamiento, así que esa variable casi nunca existe donde se necesita.
- **Huber:** no cambia nada frente al modelo actual.

Las casas siguen cerca de 31% con todas las variantes. Ajustar el modelo no resuelve eso. Falta
información, como el área del lote. Elegimos la pérdida MAE.

## Validación aleatoria y por zonas

El error mediano de 21,0% del capítulo anterior es menor que el de estas 20 divisiones. Revisamos
por qué, y agregamos una validación aleatoria para ver cuánto cambia el resultado según el diseño.

In [9]:
y_ent = ent.log_precio.to_numpy()

def fuera_de_muestra(particion):
    p = np.empty(len(ent))
    for a, b in particion:
        p[b] = ajustar(ent.iloc[a], ent.iloc[b], BASE)
    return p

p_aleatoria = fuera_de_muestra(KFold(5, shuffle=True, random_state=42).split(ent))
p_bloques = fuera_de_muestra(GroupKFold(5).split(ent, groups=ent.bloque))
bloque_alameda = ent.loc[(ent.barrio == "SECTOR ALAMEDA DEL RIO").fillna(False).to_numpy(dtype=bool), "bloque"].value_counts().index[0]
en_alameda = (ent.bloque == bloque_alameda).to_numpy()
lat, lon = ent.lat.to_numpy(), ent.lon.to_numpy()
pd.DataFrame({
    "Aleatoria": metricas_completas(y_ent, p_aleatoria, lat, lon),
    "Por bloques": metricas_completas(y_ent, p_bloques, lat, lon),
    "Por bloques, sin Alameda": metricas_completas(y_ent[~en_alameda], p_bloques[~en_alameda], lat[~en_alameda], lon[~en_alameda]),
    "Por bloques, solo Alameda": metricas_completas(y_ent[en_alameda], p_bloques[en_alameda]),
}).round(2)

,Aleatoria,Por bloques,"Por bloques, sin Alameda","Por bloques, solo Alameda"
R² (log),0.77,0.67,0.67,0.42
R² (pesos),0.48,0.39,0.37,0.40
RMSE (log10),0.16,0.19,0.19,0.15
MAE (log10),0.10,0.13,0.14,0.10
RMSE (millones),273.45,297.57,329.20,63.95
MAE (millones),72.69,96.45,109.34,41.56
Error abs. mediano (millones),31.19,47.57,52.52,28.21
MdAPE (%),13.46,21.01,22.95,16.17
MAPE (%),25.06,34.04,35.37,28.42
P90 del error (%),56.52,64.86,70.53,41.77


- **El diseño pesa mucho.** Con validación aleatoria el error mediano sería de 13,5% y el R² de 0,77;
  por zonas es de 21,0% y 0,67. La validación aleatoria es la que usan muchos estudios, así que sus
  cifras no se comparan directamente con las nuestras. Además tiene una ventaja: la misma manzana o
  el mismo edificio queda a la vez en entrenamiento y en validación. Mide cuánto acierta el modelo
  donde ya hay ventas cercanas, mientras que la validación por zonas mide cuánto acierta donde no
  las hay.
- **El 21,0% era optimista.** El bloque de Alameda del Río tiene el 19% de las ventas de
  entrenamiento y un error de 16,2%. Sin él, la validación por bloques sube a 23,0%.

## Área del lote de las casas

Al precio de una casa le pesa mucho el tamaño del lote, y la base no lo traía. Lo tomamos de la capa
de terrenos de la Gerencia de Gestión Catastral de Barranquilla (servicio ArcGIS del Distrito,
capa *Terreno*), que trae el área de cada predio con su código predial de 30 dígitos. Lo cruzamos
por `codigo_predial` y lo guardamos en `datos/terreno_casas.csv`. Solo lo usamos en casas: en los
apartamentos el área del lote es la del edificio entero.

In [10]:
terreno = pd.read_csv("datos/terreno_casas.csv", dtype={"name": str}).set_index("name").Shape__Area
for x in (ent, prueba):
    es_casa = (x.tipo_NPH == 1).to_numpy()
    x["log_terreno"] = np.log10(x.codigo_predial.astype(str).map(terreno).astype(float).where(es_casa))
CON_LOTE = BASE + ["log_terreno"]
casas_total = (ent.tipo_NPH == 1).sum() + (prueba.tipo_NPH == 1).sum()
con_lote = ent.log_terreno.notna().sum() + prueba.log_terreno.notna().sum()
print(f"Casas con área del lote: {con_lote:,} de {casas_total:,} ({con_lote / casas_total:.1%})")
print(f"Área mediana del lote: {10 ** pd.concat([ent.log_terreno, prueba.log_terreno]).median():.0f} m²")

Casas con área del lote: 3,937 de 3,948 (99.7%)
Área mediana del lote: 130 m²


In [11]:
filas = []
for s, es_val in divisiones(ent):
    tr, te = ent[~es_val].reset_index(drop=True), ent[es_val].reset_index(drop=True)
    y = te.log_precio.to_numpy()
    casa = (te.tipo_NPH == 1).to_numpy()
    for nombre, cols in [("Pérdida MAE", BASE), ("Pérdida MAE + lote", CON_LOTE)]:
        p = ajustar(tr, te, cols, **MAE)
        for grupo, mascara in [("todas", np.ones(len(y), bool)), ("casas", casa), ("apartamentos u otros", ~casa)]:
            filas.append({"division": s, "modelo": nombre, "grupo": grupo, **metricas_completas(y[mascara], p[mascara])})
lote = pd.DataFrame(filas)
lote.pivot_table(index="grupo", columns="modelo", values=["MdAPE (%)", "R² (log)", "PPE20 (%)"]).round(2)

MdAPE (%)                      PPE20 (%)                       R² (log)                   
modelo               Pérdida MAE Pérdida MAE + lote Pérdida MAE Pérdida MAE + lote Pérdida MAE Pérdida MAE + lote
grupo                                                                                                            
apartamentos u otros       21.89              22.06       47.51              47.51        0.52               0.51
casas                      31.53              29.88       33.58              35.33        0.64               0.66
todas                      24.42              24.23       43.12              43.52        0.63               0.64

In [12]:
comparacion = {}
for grupo in ["todas", "casas", "apartamentos u otros"]:
    a = lote[lote.grupo == grupo].pivot(index="division", columns="modelo", values="MdAPE (%)")
    dif_lote = a["Pérdida MAE + lote"] - a["Pérdida MAE"]
    margen = 2.093 * dif_lote.std() / np.sqrt(len(dif_lote))
    comparacion[grupo] = {"diferencia media": dif_lote.mean(), "IC95 inf": dif_lote.mean() - margen,
                          "IC95 sup": dif_lote.mean() + margen, "divisiones en que mejora": (dif_lote < 0).sum()}
pd.DataFrame(comparacion).T.round(2)

,diferencia media,IC95 inf,IC95 sup,divisiones en que mejora
todas,-0.19,-0.62,0.24,10.00
casas,-1.65,-2.38,-0.92,18.00
apartamentos u otros,0.16,-0.61,0.93,8.00


Cruzan 3.937 de las 3.948 casas (99,7%), y el lote mediano mide 130 m². En las 20 divisiones, el área del lote baja el error mediano de las casas de 31,5% a 29,9%: son 1,65 puntos (intervalo de −2,38 a −0,92) y mejora en 18 de las 20 divisiones. En los apartamentos no cambia nada. En el total la mejora es pequeña (0,19 puntos), porque las casas son cerca de tres de cada diez ventas.

Es la única variable que mejora las casas de forma consistente, así que queda en el modelo final.

## Otras pruebas de optimización

Sobre el modelo con pérdida MAE y área del lote probamos varias formas más de optimizarlo, con las
mismas 20 divisiones. Son unos 500 entrenamientos, así que no se corren aquí: están en
`optimizar_modelo.py`. El reajuste de hiperparámetros se hizo en otras 10 divisiones, para no
elegir y evaluar con las mismas.

| Prueba | Diferencia de error mediano (puntos) | Divisiones en que mejora |
|---|---|---|
| Reajustar hiperparámetros (31 combinaciones) | Ganó la combinación actual | — |
| Limitar a 300 ventas el peso de cada zona | −0,55 (−1,08 a −0,03) | 13 de 20 |
| Limitar a 150 ventas el peso de cada zona | +0,27 (−0,16 a +0,70) | 7 de 20 |
| Promediar 5 semillas | −0,24 (−0,56 a +0,08) | 14 de 20 |
| Límite de 300 y 5 semillas | −0,20 (−0,47 a +0,07) | 13 de 20 |
| LightGBM con pérdida L1 | +0,19 (−0,33 a +0,72) | 9 de 20 |
| Promedio de XGBoost y LightGBM | −0,04 (−0,32 a +0,25) | 13 de 20 |
| Coordenadas rotadas | +0,42 (−0,36 a +1,20) | 7 de 20 |

Ninguna se queda. Los hiperparámetros del capítulo anterior siguen siendo los mejores. Limitar el
peso de cada zona parecía ayudar, pero es azar: con 150 ventas empeora, y al promediar 5 semillas
el error es el mismo con límite o sin él (24,0%). Solo cambiar la semilla mueve el error unos 0,3
puntos, así que diferencias de ese tamaño no significan nada. LightGBM y las coordenadas rotadas
no ayudan.

Con los datos actuales, el modelo está cerca de su techo. Para bajar más el error hace falta
información nueva.

## Prueba final

Evaluamos una sola vez el modelo elegido, XGBoost con pérdida MAE y área del lote, junto con los
modelos anteriores y las dos referencias, en las zonas que dejamos aparte. Con esas zonas
calculamos también intervalos: remuestreamos los bloques de prueba 2.000 veces.

In [13]:
y_prueba = prueba.log_precio.to_numpy()
la, lo = prueba.lat.to_numpy(), prueba.lon.to_numpy()
predicciones = {"Regla ingenua": ingenua(ent, prueba), "Regresión lineal": lineal(ent, prueba),
                "XGBoost actual": ajustar(ent, prueba, BASE),
                "XGBoost con pérdida MAE": ajustar(ent, prueba, BASE, **MAE),
                "Modelo final: MAE + lote": ajustar(ent, prueba, CON_LOTE, **MAE)}
pd.DataFrame({k: metricas_completas(y_prueba, p, la, lo) for k, p in predicciones.items()}).round(2)

,Regla ingenua,Regresión lineal,XGBoost actual,XGBoost con pérdida MAE,Modelo final: MAE + lote
R² (log),0.31,0.57,0.63,0.64,0.67
R² (pesos),-0.04,0.54,0.58,0.60,0.61
RMSE (log10),0.28,0.22,0.21,0.20,0.20
MAE (log10),0.20,0.16,0.15,0.14,0.14
RMSE (millones),291.54,193.46,184.46,180.14,177.37
MAE (millones),138.13,98.08,91.03,86.33,84.64
Error abs. mediano (millones),68.18,50.66,45.05,43.62,41.85
MdAPE (%),28.10,26.79,25.07,21.64,20.23
MAPE (%),66.33,38.98,34.04,32.83,32.33
P90 del error (%),172.31,80.87,69.01,69.53,65.06


In [14]:
casa = (prueba.tipo_NPH == 1).to_numpy()
XGB = ["XGBoost actual", "XGBoost con pérdida MAE", "Modelo final: MAE + lote"]
pd.concat({g: pd.DataFrame({k: metricas_completas(y_prueba[m], predicciones[k][m]) for k in XGB})
           for g, m in [("casas", casa), ("apartamentos u otros", ~casa)]}, axis=1).round(2)

casas                                                  apartamentos u otros                          \
                              XGBoost actual XGBoost con pérdida MAE Modelo final: MAE + lote       XGBoost actual XGBoost con pérdida MAE   
R² (log)                                0.52                    0.50                     0.53                 0.66                    0.69   
R² (pesos)                              0.40                    0.43                     0.50                 0.62                    0.63   
RMSE (log10)                            0.24                    0.24                     0.23                 0.19                    0.18   
MAE (log10)                             0.19                    0.19                     0.19                 0.13                    0.12   
RMSE (millones)                       169.71                  165.94                   155.80               192.19                  187.59   
MAE (millones)                         88.71                   87.80                    83.86                92.32                   85.50   
Error abs. mediano (millones)          50.03                   50.83                    52.86                41.30                   37.36   
MdAPE (%)                              34.12                   34.01                    34.76                22.23                   17.63   
MAPE (%)                               43.77                   43.49                    43.66                28.62                   26.88   
P90 del error (%)                      87.86                   89.46                    87.45                58.19                   56.35   
PPE10 (%)                              15.95                   15.48                    14.55                25.11                   29.66   
PPE20 (%)                              30.62                   29.57                    30.62                45.68                   57.50   
PPE30 (%)                              44.24                   43.54                    42.37                69.18                   72.68   
Sesgo mediano (%)                     -10.50                   -9.39                   -10.43               -10.41                   -7.69   
Razón mediana                           0.90                    0.91                     0.90                 0.90                    0.92   
COD                                    47.82                   47.04                    47.64                30.45                   27.71   
PRD                                     1.23                    1.24                     1.24                 1.12                    1.11   
PRB                                    -0.11                   -0.13                    -0.13                -0.01                   -0.01   

                                                        
                              Modelo final: MAE + lote  
R² (log)                                          0.72  
R² (pesos)                                        0.63  
RMSE (log10)                                      0.17  
MAE (log10)                                       0.11  
RMSE (millones)                                 188.32  
MAE (millones)                                   85.07  
Error abs. mediano (millones)                    34.83  
MdAPE (%)                                        16.92  
MAPE (%)                                         26.01  
P90 del error (%)                                54.90  
PPE10 (%)                                        30.50  
PPE20 (%)                                        60.29  
PPE30 (%)                                        74.76  
Sesgo mediano (%)                                -5.69  
Razón mediana                                     0.94  
COD                                              26.86  
PRD                                               1.13  
PRB                                              -0.05

In [15]:
por_bloque = {b: np.where(prueba.bloque.to_numpy() == b)[0] for b in prueba.bloque.unique()}
nombres_bloque = list(por_bloque)
mdape = lambda p, i: np.median(np.abs(10.0 ** (p[i] - y_prueba[i]) - 1)) * 100
pa, pm, pf = (predicciones[k] for k in XGB)
rng = np.random.default_rng(42)
remuestras = []
for _ in range(2000):
    i = np.concatenate([por_bloque[b] for b in rng.choice(nombres_bloque, len(nombres_bloque))])
    ic = i[casa[i]]
    remuestras.append((mdape(pa, i), mdape(pm, i), mdape(pf, i), mdape(pm, i) - mdape(pa, i),
                       mdape(pf, i) - mdape(pm, i), mdape(pf, ic) - mdape(pm, ic)))
remuestras = np.array(remuestras)
columnas = ["MdAPE XGBoost actual", "MdAPE pérdida MAE", "MdAPE modelo final",
            "MAE menos actual", "Lote menos MAE (todas)", "Lote menos MAE (casas)"]
pd.DataFrame(np.percentile(remuestras, [2.5, 97.5], axis=0), index=["IC95 inf", "IC95 sup"], columns=columnas).round(2)

,MdAPE XGBoost actual,MdAPE pérdida MAE,MdAPE modelo final,MAE menos actual,Lote menos MAE (todas),Lote menos MAE (casas)
IC95 inf,21.27,18.88,17.54,-6.27,-2.83,-0.83
IC95 sup,29.36,26.64,25.69,0.70,0.34,3.33


En la prueba final, el modelo con área del lote tiene un error mediano de 20,2% y un R² de 0,67, frente a 25,1% y 0,63 del XGBoost del capítulo anterior. La mitad de las ventas (49,7%) queda con un error menor a 20%.

Pero la prueba no confirma de dónde viene la mejora:

- En las casas, el error mediano sube de 34,0% a 34,8%; el intervalo de esa diferencia va de −0,8 a +3,3 puntos.
- Los apartamentos, que no tienen el dato del lote, bajan de 17,6% a 16,9%.

Con solo 24 zonas la prueba es muy ruidosa, y los intervalos de las diferencias incluyen el cero. Las cifras más prudentes son las de las 20 divisiones: un error mediano de 24,2% y un R² de 0,64.

## Resumen

- **Modelo final:** XGBoost con pérdida MAE y área del lote, con los hiperparámetros del capítulo anterior. En zonas que no ha visto, su error mediano esperable es de 24,2% y su R² de 0,64, en promedio de 20 divisiones. En la prueba final da 20,2% y 0,67.
- **El azar pesa mucho:** según las zonas que queden aparte, el error mediano del modelo va de 15,5% a 31,9%.
- **Lo que funcionó:** la pérdida MAE (0,7 puntos menos de error) y el área del lote, que baja 1,7 puntos el error de las casas.
- **Lo que no funcionó:** el barrio, la pérdida Huber, los modelos separados por tipo, reajustar los hiperparámetros, limitar el peso de cada zona, promediar semillas, LightGBM y las coordenadas rotadas.
- **El 21,0% de validación del capítulo anterior era optimista,** porque incluía el bloque de Alameda del Río. Con validación aleatoria el error mediano sería de 13,5%; las cifras de los estudios que usan ese diseño no se comparan directamente con las nuestras.
- **El modelo está lejos de las referencias de la IAAO** (COD y PRD) y sobrevalora lo barato frente a lo caro. Sirve como apoyo exploratorio, no como avalúo masivo.

Con los datos actuales el modelo está cerca de su techo. Para bajar más el error hacen falta datos nuevos:

1. La capa de construcción del catastro (pisos, altura, huella y año de construcción) y las licencias de ampliación o adecuación, para describir mejor las casas.
2. El código predial de la unidad en los apartamentos que hoy tienen el del edificio, que son el 45%.
3. Confirmar si los precios del observatorio son ventas cerradas u ofertas.